# 02 — Aggregations and joins

Phase 2 of the roadmap: `groupBy` / `agg`, column expressions, and joins against the retail CSVs.

Before running: `python scripts/generate_data.py` from the project root (if you haven't already).

In [ ]:
import os

# Windows: needed before Spark starts if you write Parquet later
os.environ.setdefault("HADOOP_HOME", r"C:\hadoop")
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ.get("PATH", "")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("aggregations-and-joins")
    .master("local[*]")
    .getOrCreate()
)

DATA_DIR = "../data/generated"
print(f"Spark version: {spark.version}")

## Load the retail tables

In [ ]:
orders = spark.read.csv(f"{DATA_DIR}/orders.csv", header=True, inferSchema=True)
customers = spark.read.csv(f"{DATA_DIR}/customers.csv", header=True, inferSchema=True)
products = spark.read.csv(f"{DATA_DIR}/products.csv", header=True, inferSchema=True)

orders.printSchema()
print(f"orders={orders.count()}, customers={customers.count()}, products={products.count()}")

## Warm-up — `groupBy` + `agg`

You've seen simple `.groupBy(...).count()`. `agg` lets you compute several metrics at once, and `F.col` / `F.when` / `F.round` build richer expressions.

In [ ]:
# Example: orders and revenue by discount level
(
    orders
    .groupBy("discount")
    .agg(
        F.count("*").alias("num_orders"),
        F.round(F.sum("order_total"), 2).alias("revenue"),
        F.round(F.avg("order_total"), 2).alias("avg_order"),
    )
    .orderBy("discount")
    .show()
)

## Joins

| how | Keeps |
|-----|--------|
| `inner` | Only keys in **both** sides |
| `left` | All left rows; right cols null when no match |
| `left_anti` | Left rows whose key is **missing** on the right |

Join `orders` → `products` on `product_id`, then aggregate.

In [ ]:
# Example: revenue by product category (inner join)
revenue_by_category = (
    orders
    .join(products, on="product_id", how="inner")
    .groupBy("category")
    .agg(
        F.round(F.sum("order_total"), 2).alias("revenue"),
        F.count("order_id").alias("num_orders"),
    )
    .orderBy(F.col("revenue").desc())
)

revenue_by_category.show()

### Column expressions with `when`

Bucket orders into size bands, then count.

In [ ]:
orders_banded = orders.withColumn(
    "size_band",
    F.when(F.col("order_total") < 100, "S")
     .when(F.col("order_total") < 500, "M")
     .otherwise("L"),
)

orders_banded.groupBy("size_band").count().orderBy("size_band").show()

## Your turn

1. **Top 10 customers by lifetime spend** — join `orders` to `customers`, sum `order_total` per customer, show the top 10 (include name columns).
2. **Orphaned orders** — how many orders reference a `customer_id` that doesn't exist in `customers`? (`how="left_anti"`). What fraction of all orders is that?
3. **Revenue by state** — join orders → customers, sum revenue by `state`, order highest first.
4. **Category mix for big spenders** — among customers whose lifetime spend is over $5,000, what's the revenue breakdown by product `category`?

Hints: chain joins; use `.agg(...)`; `F.countDistinct` if you need unique customers; filter *after* aggregating for question 4 (or use a subquery-style DataFrame).

In [ ]:
# 1. Top 10 customers by lifetime spend


In [ ]:
# 2. Orphaned orders — count and fraction of all orders


In [ ]:
# 3. Revenue by state


In [ ]:
# 4. Category mix for customers with lifetime spend > $5000


When you're done:

```python
spark.stop()
```

Next: `03_unstructured_to_structured.ipynb` (parsing messy files), then Phase 3 in the README (cleaning + Spark SQL).

In [ ]:
# spark.stop()